<img src="https://datasciencedegree.wisconsin.edu/wp-content/themes/data-gulp/images/logo.svg" width="300">

# Baby Boom

This file accompanies a narrated presentation.  

---

## File information
The file ```babyboom.csv``` contains information on all 44 babies born at one hospital in one 24-hour period.  It has 3 columns:  

* Time = time of birth, in the format hhmm
* Sex = the sex of the baby
* Weight = the weight of the baby, in grams.

Dataset: “Time of Birth, Sex, and Birth Weight of 44 Babies," submitted by Peter K. Dunn, University of Southern Queensland. Dataset obtained from the Journal of Statistics Education
(http://www.amstat.org/publications/jse). Accessed 14 July 2015. 
Used by permission of author.
Metadata:  http://www.amstat.org/publications/jse/datasets/babyboom.txt


We'll use the [pandas library](http://pandas.pydata.org/) to work with this data set.

In [1]:
import pandas as pd
import numpy as np

---

## Reading the data

The pandas library includes a data structure called a Data Frame, which is optimized for working with tabular data.  We read in babyboom.csv as a data frame.

In [2]:
bb_df = pd.read_csv('babyboom.csv')
bb_df

,Time,Sex,Weight
0,1049,F,3746.00
1,1305,Male,3690.00
2,1407,F,3480.00
3,1909,boy,4162.00
4,735,M,3380.00
5,1446,F,3428.00
6,405,girl,2208.00
7,104,F,3334.00
8,909,F,3208.00
9,1053,F,3523.00


Note that blank lines have been replaced by the symbol ```NaN```, which stands for "not a number".  This special value is a constant in the ```numpy``` package, and can be accessed as ```np.nan```.

---

## Dealing with ?

We notice that the data set includes at least one question mark.  We'd like to replace all question marks by ```NaN```.  To do so, we create a function, and then apply this function to every cell in the data frame using the ```applymap``` function for data frames.

In [3]:
def remove_question(s):
    """ return np.nan if the argument is a question mark """
    if s == "?":
        return np.nan
    else:
        return s

In [6]:
bb_df = bb_df.map(remove_question)
bb_df

,Time,Sex,Weight
0,1049,F,3746.00
1,1305,Male,3690.00
2,1407,F,3480.00
3,1909,boy,4162.00
4,735,M,3380.00
5,1446,F,3428.00
6,405,girl,2208.00
7,104,F,3334.00
8,909,F,3208.00
9,1053,F,3523.00


---

## Canonicalizing the Sex column

We notice that the Sex column contains many different types of data.  We'd like to standardize this data as M or F.  The first step is to get a list of all of the different values in the column.

We access this column using its text label.  Each column of a pandas data frame is a ```Series``` object, and we can use the built-in ```unique()``` function for series.

We can use the ```describe()``` function for data frames to get a quick summary of the numerical information in the data frame.  The ```count``` function will ignore any ```NaN``` values.

In [7]:
bb_df.Sex.unique()

<StringArray>
['F', 'Male', 'boy', 'M', 'girl', 'not recorded', 'female', nan]
Length: 8, dtype: str

We want to map each of the values from the Sex column to "M" or "F".  We could do this using lots of if-then statements, but because we're describing a correspondence, it makes sense to create a dictionary.

In [8]:
sex_dict = { 'F':'F', 'Male':'M', 'boy':'M', 'M':'M', 'girl':'F', 'not recorded':np.nan, 'female':'F', np.nan:np.nan}

In [9]:
# We can use the get command to obtain the value for a specified key
sex_dict.get("girl")

'F'

In [11]:
# We create a lambda function which calls our dictionary, and map it to every element of the Sex column
# This returns a Series object
bb_df['Sex'].map(lambda s: sex_dict.get(s))

0       F
1       M
2       F
3       M
4       M
5       F
6       F
7       F
8       F
9       F
10      M
11      F
12      M
13      M
14      M
15      F
16      F
17      M
18      M
19    NaN
20      M
21      F
22      F
23      M
24      F
25      F
26      F
27      M
28      M
29      M
30      M
31      M
32      M
33      M
34      M
35      F
36      M
37      M
38      M
39    NaN
40      F
41      F
42      M
43      M
Name: Sex, dtype: str

In [12]:
# Of course, what we really want to do is REPLACE the Sex column in our data frame

bb_df['Sex'] = bb_df['Sex'].map(lambda s: sex_dict.get(s))
bb_df

,Time,Sex,Weight
0,1049,F,3746.00
1,1305,M,3690.00
2,1407,F,3480.00
3,1909,M,4162.00
4,735,M,3380.00
5,1446,F,3428.00
6,405,F,2208.00
7,104,F,3334.00
8,909,F,3208.00
9,1053,F,3523.00


Now we can group our data frame by M or F, and compare descriptive statistics for each group.

In [13]:
bb_group = bb_df.groupby('Sex')
bb_group.describe()

Weight                                                                 
     count         mean         std   min      25%     50%      75%     max
Sex                                                                        
F     18.0  2938.428889  961.998214  7.72  2431.25  3306.0  3512.25  3866.0
M     24.0  3296.192917  778.415345  3.63  3262.00  3404.0  3641.25  4162.0

---

## Weight conversion

We've demonstrated using the ```map``` command on Series to transform columns of our data frame.  We can use the ```apply``` command to do operations on rows.

In [14]:
# A function to apply to a row of our data frame

def convert_to_lb(row):
    """ convert weight in grams to weight in pounds"""
    return row['Weight']/453.59237

In [17]:
# Specifying axis=1 applies the function to each row; the default would apply the function to each column
kg_to_lb = lambda row: row['Weight']/453.59237
bb_df['Weight in lb'] = bb_df.apply(kg_to_lb, axis=1)
bb_df

,Time,Sex,Weight,Weight in lb
0,1049,F,3746.00,8.258516
1,1305,M,3690.00,8.135057
2,1407,F,3480.00,7.672087
3,1909,M,4162.00,9.175639
4,735,M,3380.00,7.451624
5,1446,F,3428.00,7.557446
6,405,F,2208.00,4.867807
7,104,F,3334.00,7.350212
8,909,F,3208.00,7.072429
9,1053,F,3523.00,7.766885


Exercise for reader: 

What would happen if we didn't specify `axis=1`?

In [18]:
# your code here
kg_to_lb = lambda row: row['Weight']/453.59237
bb_df['Weight in lb'] = bb_df.apply(kg_to_lb)
bb_df

KeyError: 'Weight'

---

## Filtering

Now, let's work with only the bigger babies -- those with weight at least 8 pounds.  Pandas makes this easy:

In [19]:
bb_df[bb_df["Weight in lb"]>8]

,Time,Sex,Weight,Weight in lb
0,1049,F,3746.0,8.258516
1,1305,M,3690.0,8.135057
3,1909,M,4162.0,9.175639
16,2217,F,3866.0,8.523071
27,1514,M,3783.0,8.340087
30,155,M,3838.0,8.461342
35,5,F,3837.0,8.459137
38,1256,M,3920.0,8.642121
42,2037,M,3736.0,8.236470


In [20]:
big_babies = bb_df[bb_df["Weight in lb"]>8]
big_babies.reset_index(drop=True)

,Time,Sex,Weight,Weight in lb
0,1049,F,3746.0,8.258516
1,1305,M,3690.0,8.135057
2,1909,M,4162.0,9.175639
3,2217,F,3866.0,8.523071
4,1514,M,3783.0,8.340087
5,155,M,3838.0,8.461342
6,5,F,3837.0,8.459137
7,1256,M,3920.0,8.642121
8,2037,M,3736.0,8.236470


---

## Saving our data

We'd like to save our data for possible analysis in R.

Because R won't recognize the ```NaN``` symbol, we replace null data with an empty string.

In [21]:
# We use the inplace flag to change the original data frame, rather than creating a new one
bb_df.fillna("", inplace=True)
bb_df

,Time,Sex,Weight,Weight in lb
0,1049,F,3746.00,8.258516
1,1305,M,3690.00,8.135057
2,1407,F,3480.00,7.672087
3,1909,M,4162.00,9.175639
4,735,M,3380.00,7.451624
5,1446,F,3428.00,7.557446
6,405,F,2208.00,4.867807
7,104,F,3334.00,7.350212
8,909,F,3208.00,7.072429
9,1053,F,3523.00,7.766885


In [22]:
# Save the data frame to a .csv file
# Specifying index=False prevents us from writing a column of row numbers

bb_df.to_csv("babyboom_clean.csv", index=False)